# 05 - Bronze EDA and profiling
Read-only. It reports **observed** values: nothing is assumed, changed or fixed here.
Use the results to fill your DQ register and rule catalogue, and decide what Silver should do (see the decision list at the end).

In [0]:
b = "capstone6_telecom.bronze"

k = f"{b}.bronze_network_kpi_measurements"
o = f"{b}.bronze_network_outage_events"
q = f"{b}.bronze_customer_complaints"
t = f"{b}.bronze_tower_master"
c = f"{b}.bronze_network_topology_master"
r = f"{b}.bronze_region_lookup"

keys = {k: "measurement_id", o: "outage_id", q: "complaint_id", t: "tower_id", c: "cell_id", r: "region_id"}
AUDIT = ["source_file", "source_file_modified", "ingest_ts", "row_hash", "source_system", "_rescued_data"]

## 1. Source inventory and completeness
Rows loaded versus rows in the source files.

In [0]:
display(spark.sql(f"""
SELECT 'network_kpi_measurements' AS dataset, COUNT(*) AS loaded_rows, 11894 AS source_file_rows FROM {k}
UNION ALL SELECT 'network_outage_events',      COUNT(*), 696   FROM {o}
UNION ALL SELECT 'customer_complaints',        COUNT(*), 354   FROM {q}
UNION ALL SELECT 'tower_master',               COUNT(*), 161   FROM {t}
UNION ALL SELECT 'network_topology_master',    COUNT(*), 170   FROM {c}
UNION ALL SELECT 'region_lookup',              COUNT(*), 179   FROM {r}
"""))
display(spark.sql(f"SELECT source_file, COUNT(*) AS rows FROM {k} GROUP BY source_file ORDER BY source_file"))
display(spark.sql(f"DESCRIBE TABLE {k}"))

## 2. Key uniqueness and exact duplicates
`row_hash` covers all source columns: same key + same hash is a repeated delivery, same key + different hash is a changed version.

In [0]:
parts = []
for tbl, key in keys.items():
    parts.append(f"""SELECT '{tbl.split('.')[-1]}' AS dataset, '{key}' AS business_key,
        COUNT(*) AS rows, COUNT(DISTINCT {key}) AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT {key}) AS rows_over_distinct_keys,
        COUNT(*) - COUNT(DISTINCT row_hash) AS exact_duplicate_rows,
        SUM(CASE WHEN {key} IS NULL OR trim({key}) = '' THEN 1 ELSE 0 END) AS blank_keys
        FROM {tbl}""")
display(spark.sql(" UNION ALL ".join(parts)))

In [0]:
# which keys repeat in the KPI feed, how many copies, which files, and are the copies identical?
display(spark.sql(f"""
SELECT measurement_id, COUNT(*) AS copies, COUNT(DISTINCT row_hash) AS distinct_versions,
       collect_set(regexp_extract(source_file, '(batch_[0-9]+)', 1)) AS batches
FROM {k} GROUP BY measurement_id HAVING COUNT(*) > 1 ORDER BY measurement_id"""))

## 3. Blank values per column

In [0]:
for tbl in keys:
    cols = [x for x in spark.table(tbl).columns if x not in AUDIT]
    sel = ", ".join([f"SUM(CASE WHEN {x} IS NULL OR trim({x}) = '' THEN 1 ELSE 0 END) AS {x}" for x in cols])
    print(tbl.split(".")[-1])
    display(spark.sql(f"SELECT {sel} FROM {tbl}"))

## 4. Category frequencies (low-cardinality columns)

In [0]:
cats = [(k, "network_generation"), (k, "active_sessions"),
        (o, "outage_type"), (o, "severity"), (o, "root_cause_code"), (o, "affected_cells"), (o, "incident_status"),
        (q, "complaint_type"), (q, "severity"), (q, "channel"), (q, "ticket_status"),
        (t, "site_type"), (t, "network_generation"), (t, "tower_status"),
        (c, "frequency_band"), (c, "sector"), (c, "technology"), (c, "active_flag"), (c, "vendor"),
        (r, "state"), (r, "country"), (r, "service_zone"), (r, "sla_tier"), (r, "active_flag")]
parts = [f"SELECT '{tbl.split('.')[-1]}' AS dataset, '{col}' AS column_name, {col} AS value, COUNT(*) AS rows FROM {tbl} GROUP BY {col}"
         for tbl, col in cats]
display(spark.sql(" UNION ALL ".join(parts) + " ORDER BY dataset, column_name, rows DESC"))

## 5. KPI numeric columns: type check, range and percentiles

In [0]:
nums = ["latency_ms", "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct"]
parts = [f"""SELECT '{n}' AS column_name,
        SUM(CASE WHEN {n} IS NOT NULL AND trim({n}) <> '' AND try_cast({n} AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS not_a_number,
        SUM(CASE WHEN try_cast({n} AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS negative_values,
        MIN(try_cast({n} AS DOUBLE)) AS min_value,
        PERCENTILE(try_cast({n} AS DOUBLE), 0.01) AS p01,
        PERCENTILE(try_cast({n} AS DOUBLE), 0.50) AS median,
        PERCENTILE(try_cast({n} AS DOUBLE), 0.99) AS p99,
        MAX(try_cast({n} AS DOUBLE)) AS max_value
        FROM {k}""" for n in nums]
display(spark.sql(" UNION ALL ".join(parts)))

In [0]:
# unusual latency values, and how the normal range looks without them
display(spark.sql(f"""
SELECT latency_ms, COUNT(*) AS rows FROM {k}
WHERE try_cast(latency_ms AS DOUBLE) < 0 OR try_cast(latency_ms AS DOUBLE) > 1000
GROUP BY latency_ms ORDER BY rows DESC"""))
display(spark.sql(f"""
SELECT MIN(v) AS min_normal, PERCENTILE(v, 0.5) AS median_normal, MAX(v) AS max_normal FROM (
  SELECT try_cast(latency_ms AS DOUBLE) AS v FROM {k}) WHERE v >= 0 AND v <= 1000"""))

## 6. Timestamps: parsing, ranges, ingestion lag, batch ordering

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS rows,
       SUM(CASE WHEN try_to_timestamp(measurement_ts) IS NULL THEN 1 ELSE 0 END) AS unparseable_measurement_ts,
       SUM(CASE WHEN try_to_timestamp(ingestion_ts) IS NULL THEN 1 ELSE 0 END) AS unparseable_ingestion_ts,
       MIN(try_to_timestamp(measurement_ts)) AS first_measurement, MAX(try_to_timestamp(measurement_ts)) AS last_measurement,
       MIN((unix_timestamp(try_to_timestamp(ingestion_ts)) - unix_timestamp(try_to_timestamp(measurement_ts))) / 60) AS min_lag_minutes,
       MAX((unix_timestamp(try_to_timestamp(ingestion_ts)) - unix_timestamp(try_to_timestamp(measurement_ts))) / 60) AS max_lag_minutes
FROM {k}"""))

# time range inside each batch file: a batch that reaches back before the previous batch holds late or replayed rows
display(spark.sql(f"""
SELECT regexp_extract(source_file, '(batch_[0-9]+)', 1) AS batch, COUNT(*) AS rows,
       MIN(try_to_timestamp(measurement_ts)) AS first_ts, MAX(try_to_timestamp(measurement_ts)) AS last_ts
FROM {k} GROUP BY 1 ORDER BY 1"""))

In [0]:
# rows in the last batch whose measurement_id already arrived in an earlier batch
display(spark.sql(f"""
WITH x AS (SELECT measurement_id, regexp_extract(source_file, '(batch_[0-9]+)', 1) AS batch FROM {k})
SELECT batch, COUNT(*) AS rows_repeating_earlier_ids
FROM x WHERE EXISTS (SELECT 1 FROM x y WHERE y.measurement_id = x.measurement_id AND y.batch < x.batch)
GROUP BY batch ORDER BY batch"""))

## 7. Reference coverage: orphan keys
`orphans` are non-blank values with no match in the parent table. `blank` values are counted separately (a missing optional attribute is not an orphan).

In [0]:
def fk(child, col, parent, pcol, label):
    return f"""SELECT '{label}' AS relationship, COUNT(*) AS child_rows,
        SUM(CASE WHEN ch.{col} IS NULL OR trim(ch.{col}) = '' THEN 1 ELSE 0 END) AS blank,
        SUM(CASE WHEN ch.{col} IS NOT NULL AND trim(ch.{col}) <> '' AND p.{pcol} IS NULL THEN 1 ELSE 0 END) AS orphans
        FROM {child} ch LEFT JOIN (SELECT DISTINCT {pcol} FROM {parent}) p ON ch.{col} = p.{pcol}"""

parts = [fk(k, "tower_id", t, "tower_id", "kpi.tower_id -> tower_master"),
         fk(k, "cell_id",  c, "cell_id",  "kpi.cell_id -> topology"),
         fk(k, "region_id", r, "region_id", "kpi.region_id -> region_lookup"),
         fk(c, "tower_id", t, "tower_id", "topology.tower_id -> tower_master"),
         fk(c, "region_id", r, "region_id", "topology.region_id -> region_lookup"),
         fk(t, "region_id", r, "region_id", "tower_master.region_id -> region_lookup"),
         fk(o, "tower_id", t, "tower_id", "outage.tower_id -> tower_master"),
         fk(q, "tower_id", t, "tower_id", "complaints.tower_id -> tower_master"),
         fk(q, "region_id", r, "region_id", "complaints.region_id -> region_lookup")]
display(spark.sql(" UNION ALL ".join(parts)))

## 8. Do the sources agree with each other? (decide which one is the source of truth)

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS kpi_rows_with_cell,
       ROUND(100 * AVG(CASE WHEN kp.tower_id  = tp.tower_id  THEN 1 ELSE 0 END), 1) AS pct_tower_matches_topology,
       ROUND(100 * AVG(CASE WHEN kp.region_id = tp.region_id THEN 1 ELSE 0 END), 1) AS pct_region_matches_topology
FROM {k} kp JOIN {c} tp ON kp.cell_id = tp.cell_id"""))

display(spark.sql(f"""
SELECT COUNT(*) AS kpi_rows,
       ROUND(100 * AVG(CASE WHEN kp.region_id = tm.region_id THEN 1 ELSE 0 END), 1) AS pct_region_matches_tower_master,
       ROUND(100 * AVG(CASE WHEN kp.network_generation = tm.network_generation THEN 1 ELSE 0 END), 1) AS pct_generation_matches_tower_master
FROM {k} kp JOIN {t} tm ON kp.tower_id = tm.tower_id"""))

display(spark.sql(f"""
SELECT COUNT(*) AS towers_in_topology,
       SUM(CASE WHEN regions > 1 THEN 1 ELSE 0 END) AS towers_with_more_than_one_region
FROM (SELECT tower_id, COUNT(DISTINCT region_id) AS regions FROM {c} GROUP BY tower_id)"""))

display(spark.sql(f"""
SELECT ROUND(100 * AVG(CASE WHEN q.region_id = tm.region_id THEN 1 ELSE 0 END), 1) AS pct_complaint_region_matches_tower_master
FROM {q} q JOIN {t} tm ON q.tower_id = tm.tower_id"""))

## 9. Business state: readings from inactive towers and cells

In [0]:
display(spark.sql(f"""
SELECT tm.tower_status, COUNT(*) AS kpi_rows FROM {k} kp JOIN {t} tm ON kp.tower_id = tm.tower_id
GROUP BY 1 ORDER BY 2 DESC"""))
display(spark.sql(f"""
SELECT tp.active_flag AS cell_active_flag, COUNT(*) AS kpi_rows FROM {k} kp JOIN {c} tp ON kp.cell_id = tp.cell_id
GROUP BY 1 ORDER BY 2 DESC"""))

## 10. Events: chronology and lifecycle

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS outages,
       MIN(try_to_timestamp(outage_start_ts)) AS first_start, MAX(try_to_timestamp(outage_start_ts)) AS last_start,
       SUM(CASE WHEN try_to_timestamp(outage_start_ts) IS NULL OR try_to_timestamp(outage_end_ts) IS NULL
                  OR try_to_timestamp(reported_ts) IS NULL THEN 1 ELSE 0 END) AS unparseable_timestamps,
       SUM(CASE WHEN try_to_timestamp(outage_end_ts) < try_to_timestamp(outage_start_ts) THEN 1 ELSE 0 END) AS end_before_start,
       SUM(CASE WHEN try_to_timestamp(reported_ts) < try_to_timestamp(outage_start_ts) THEN 1 ELSE 0 END) AS reported_before_start,
       MIN((unix_timestamp(try_to_timestamp(outage_end_ts)) - unix_timestamp(try_to_timestamp(outage_start_ts))) / 60) AS min_duration_min,
       MAX((unix_timestamp(try_to_timestamp(outage_end_ts)) - unix_timestamp(try_to_timestamp(outage_start_ts))) / 60) AS max_duration_min
FROM {o}"""))

display(spark.sql(f"""
SELECT COUNT(*) AS complaints,
       MIN(try_to_timestamp(complaint_ts)) AS first_complaint, MAX(try_to_timestamp(complaint_ts)) AS last_complaint,
       SUM(CASE WHEN try_to_timestamp(resolved_ts) < try_to_timestamp(complaint_ts) THEN 1 ELSE 0 END) AS resolved_before_complaint,
       COUNT(DISTINCT customer_id) AS distinct_customers
FROM {q}"""))

# ticket status versus resolved_ts
display(spark.sql(f"""
SELECT ticket_status, SUM(CASE WHEN resolved_ts IS NULL OR trim(resolved_ts) = '' THEN 1 ELSE 0 END) AS without_resolved_ts,
       SUM(CASE WHEN resolved_ts IS NOT NULL AND trim(resolved_ts) <> '' THEN 1 ELSE 0 END) AS with_resolved_ts
FROM {q} GROUP BY ticket_status"""))

## 11. Outage-to-telemetry overlap
How many KPI readings for the same tower fall inside an outage window, or within a few minutes of one? Change `window_min` to test your correlation window.

In [0]:
window_min = 30

display(spark.sql(f"""
SELECT (SELECT MIN(try_to_timestamp(measurement_ts)) FROM {k}) AS kpi_first,
       (SELECT MAX(try_to_timestamp(measurement_ts)) FROM {k}) AS kpi_last,
       (SELECT MIN(try_to_timestamp(outage_start_ts)) FROM {o}) AS outage_first,
       (SELECT MAX(try_to_timestamp(outage_end_ts)) FROM {o}) AS outage_last,
       (SELECT MIN(try_to_timestamp(complaint_ts)) FROM {q}) AS complaint_first,
       (SELECT MAX(try_to_timestamp(complaint_ts)) FROM {q}) AS complaint_last"""))

display(spark.sql(f"""
SELECT COUNT(*) AS kpi_readings_matched,
       COUNT(DISTINCT o.outage_id) AS outages_with_a_reading,
       SUM(CASE WHEN try_to_timestamp(kp.measurement_ts) BETWEEN try_to_timestamp(o.outage_start_ts) AND try_to_timestamp(o.outage_end_ts)
                THEN 1 ELSE 0 END) AS readings_inside_outage
FROM {o} o JOIN {k} kp ON kp.tower_id = o.tower_id
 AND try_to_timestamp(kp.measurement_ts) BETWEEN try_to_timestamp(o.outage_start_ts) - INTERVAL {window_min} MINUTES
                                              AND try_to_timestamp(o.outage_end_ts)   + INTERVAL {window_min} MINUTES"""))

## 12. Save the load-completeness evidence

In [0]:
from pyspark.sql import functions as F

n = lambda tbl: spark.table(tbl).count()
rows = [("completeness", "network_kpi_measurements", n(k), 11894),
        ("completeness", "network_outage_events", n(o), 696),
        ("completeness", "customer_complaints", n(q), 354),
        ("completeness", "tower_master", n(t), 161),
        ("completeness", "network_topology_master", n(c), 170),
        ("completeness", "region_lookup", n(r), 179),
        ("completeness", "kpi batch files loaded", spark.table(k).select("source_file").distinct().count(), 12)]

(spark.createDataFrame([(a, b_, int(x), int(y), "PASS" if x == y else "FAIL") for a, b_, x, y in rows],
                       "test_id string, metric string, actual long, expected long, result string")
      .withColumn("checked_at", F.current_timestamp())
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{b}.bronze_profiling_summary"))
display(spark.table(f"{b}.bronze_profiling_summary"))

## What to record next
Write each observation into your **DQ register** (what you saw, which rule will handle it) and decide, with your mentor:

1. Which source is the truth for a tower's region (topology cell, tower master or the KPI row)?
2. How to treat unusual latency values (invalid and nulled, or quarantined).
3. What to do with readings from inactive towers or cells.
4. What to do with open tickets that carry a resolved time.
5. How wide the outage correlation window should be.
6. Where the SLA thresholds come from (`sla_tier` holds codes only).